# Boiler B-501 Specific Fuel Consumption - Problem 1 (tasks 1.1 to 1.6, Q1 to Q3)
One step per cell, the result table after each cell. Run with Restart and Run All, then save. Outputs go to `output/`.

In [ ]:
# Cell 1: imports, constants and every decision (named constants with their source)
import re, sys                                   # re for text cleaning, sys for the python version
from pathlib import Path                         # relative paths, raw files stay read-only
import numpy as np                               # numpy for flags and statistics
import pandas as pd                              # pandas does the table work
import matplotlib.pyplot as plt                  # pyplot interface for the charts
DATA_DIR = Path(".")                             # exam files sit in the notebook folder
OUT = Path("output"); OUT.mkdir(exist_ok=True)   # all written files go to output/
print("python", sys.version.split()[0], "| pandas", pd.__version__)   # versions for the paste-back
PRICE_SHEET, LOG_SHEET = "FuelPrice", "ShiftLog" # sheet names from the handout file table
IQR_K = 1.5                                      # handout 1.5: Lower = Q1 - 1.5 IQR, Upper = Q3 + 1.5 IQR
STOP_RULE = "every hour the ShiftLog window touches"   # decision Q1 (option C), confirmed by the invigilator
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 30)   # wider tables

## 1.1 Import, standardise, append
The April file has other column names, steam in kg/h (not t/h), day/month text dates and fuel gas as text with thousands commas. Fix these, use the March names, append.

In [ ]:
# Cell 2: import both logs and look at them before changing anything
mar = pd.read_excel(DATA_DIR / "B501_2026_03.xlsx", sheet_name="Export")   # March log, sheet Export
apr = pd.read_excel(DATA_DIR / "B501_2026_04.xlsx", sheet_name="Export")   # April log, sheet Export
print("March columns:", list(mar.columns))      # March names are the target names
print("April columns:", list(apr.columns))      # April names differ
print("April dtypes:", apr.dtypes.astype(str).to_dict())   # shows fuel gas is text and the time is text
apr.head()                                      # result table: first April rows

In [ ]:
# Cell 3: standardise April to the March names and units, then append
NUM = list(mar.columns[1:6])                     # the five numeric tags (March names)
assert list(apr.columns) == ["DateTime", "Steam_kgh", "FuelGas_Nm3h", "Psteam_barg", "Tfw_C", "Tstack_C", "Fuel"]   # order is as expected
apr = apr.copy()                                 # work on a copy
apr.columns = list(mar.columns)                  # use the March column names for April (same order, checked above)
apr["Timestamp"] = pd.to_datetime(apr["Timestamp"], format="%d/%m/%Y %H:%M")   # April dates are day/month/year (31/03 proves it)
apr[NUM[0]] = apr[NUM[0]] / 1000.0               # kg/h to t/h so it matches the March unit
apr[NUM[1]] = pd.to_numeric(apr[NUM[1]].astype(str).str.replace(",", "", regex=False), errors="coerce")   # "1,008.7" to 1008.7
mar["SourceFile"] = "B501_2026_03.xlsx"          # keep where each row came from
apr["SourceFile"] = "B501_2026_04.xlsx"          # same for April
data = pd.concat([mar, apr], ignore_index=True)  # append the two tables
ROWS_APPENDED = len(data)                        # blank 1 of the template
print("rows March", len(mar), "| rows April", len(apr), "| rows after append", ROWS_APPENDED)   # proof
print("median steam t/h by file:", data.groupby("SourceFile")[NUM[0]].median().round(2).to_dict())   # same scale in both files
data.head()                                      # result table

## 1.2 Clean text, remove exact duplicates, join prices

In [ ]:
# Cell 4: FuelSource to exactly two spellings
def clean_fuel(s):                               # map any spelling to Pipeline or LNG
    k = re.sub(r"[^a-z]", "", str(s).lower())    # drop spaces, dots, hyphens, lower case
    return {"pipeline": "Pipeline", "lng": "LNG"}[k]   # KeyError if a new spelling appears
print("spellings before:", sorted(data["FuelSource"].unique()))   # list the raw spellings
data["FuelSource"] = data["FuelSource"].map(clean_fuel)   # apply the cleaning
print("spellings after:", sorted(data["FuelSource"].unique()))  # must be exactly two
assert set(data["FuelSource"]) == {"Pipeline", "LNG"}   # exactly two spellings
data["FuelSource"].value_counts().to_frame()     # result table: rows per source

In [ ]:
# Cell 5: remove exact duplicate rows (all data columns, the source file name is ignored)
KEYCOLS = ["Timestamp"] + NUM + ["FuelSource"]   # the columns that define "exact duplicate"
for c in NUM:                                    # round away float noise from kg/h / 1000
    data[c] = data[c].round(6)                   # 6 decimals is far below the data precision
dups = data[data.duplicated(KEYCOLS, keep=False)].sort_values("Timestamp")   # show the copies first
print("rows involved in duplicates:", len(dups))  # proof
data = data.drop_duplicates(KEYCOLS, keep="first").sort_values("Timestamp").reset_index(drop=True)   # keep first copy, sort by time
ROWS_AFTER_DUP = len(data)                       # blank 2 of the template
assert data["Timestamp"].is_unique               # no repeated hour is left
print("rows after removing duplicates:", ROWS_AFTER_DUP)   # proof
dups                                             # result table: the duplicate rows (31 Mar 21:00-23:00)

In [ ]:
# Cell 6: join FuelPrice (Left Outer on FuelSource) and check the join
price = pd.read_excel(DATA_DIR / "B501_Master.xlsx", sheet_name=PRICE_SHEET)[["FuelSource", "Price_THB_per_Nm3"]]   # key and price
assert price["FuelSource"].is_unique             # a repeated key would multiply rows
n0 = len(data)                                   # rows before the join
data = data.merge(price, on="FuelSource", how="left", indicator=True, validate="m:1")   # left outer join with indicator
print("rows before", n0, "| after", len(data))   # row count must not change
print(data["_merge"].astype(str).value_counts().to_dict())   # every row should be matched ("both")
print("unmatched rows:", int((data["_merge"].astype(str) != "both").sum()), "| missing price:", int(data["Price_THB_per_Nm3"].isna().sum()))   # must be 0
assert len(data) == n0 and data["Price_THB_per_Nm3"].notna().all()   # join check passes
data = data.drop(columns="_merge")               # indicator no longer needed
data.groupby("FuelSource")["Price_THB_per_Nm3"].agg(["count", "mean"])   # result table

## 1.3 Missing data
Flag LongGap first (all five tags blank), then Fill Down the short gaps (LongGap rows are never filled or used).

In [ ]:
# Cell 7: flag the long gap before any filling
data["LongGap"] = data[NUM].isna().all(axis=1)   # TRUE when all five numeric tags are blank
LONGGAP_ROWS = int(data["LongGap"].sum())        # blank for the template and answer sheet
print("blank cells per tag before fill:", data[NUM].isna().sum().to_dict())   # missing count per tag
print("LongGap rows:", LONGGAP_ROWS, "| from", data.loc[data["LongGap"], "Timestamp"].min(), "to", data.loc[data["LongGap"], "Timestamp"].max())   # proof
data[data["LongGap"]].head(10)                   # result table: the long gap rows

In [ ]:
# Cell 8: Fill Down the short gaps (table is sorted by Timestamp), never inside LongGap rows
data = data.sort_values("Timestamp").reset_index(drop=True)   # make sure the sort is by time
blank_before = data[NUM].isna().mul(~data["LongGap"], axis=0).astype(bool)   # blanks outside the long gap
for c in NUM:                                    # one tag at a time
    filled = data[c].where(~data["LongGap"]).ffill()   # previous value, long-gap rows do not feed it
    data[c] = data[c].where(data[c].notna() | data["LongGap"], filled)   # fill only blanks outside the long gap
print("short-gap cells filled:", blank_before.sum().to_dict())   # how many per tag
print("blanks left outside the LongGap rows:", int(data.loc[~data["LongGap"], NUM].isna().sum().sum()))   # must be 0
assert data.loc[~data["LongGap"], NUM].notna().all().all()   # nothing missing outside the long gap
data.loc[blank_before.any(axis=1), ["Timestamp"] + NUM]   # result table: the rows that were filled

## 1.4 Condition outliers: the stop period
Chart of FI-501 against Timestamp, then read the ShiftLog for the exact start and end.

In [ ]:
# Cell 9: plot steam flow against Timestamp
fig, ax = plt.subplots(figsize=(11, 4))          # one wide chart
ax.plot(data["Timestamp"], data[NUM[0]], lw=0.7, color="tab:blue")   # steam flow line
ax.set_title("FI-501 steam flow vs Timestamp, B-501, Mar-Apr 2026")   # title
ax.set_xlabel("Timestamp"); ax.set_ylabel("Steam flow FI-501 [t/h]")   # axis labels with unit
ax.grid(alpha=0.3)                               # light grid
fig.autofmt_xdate()                              # readable date labels
fig.savefig(OUT / "chart_1_4_steam_flow.png", dpi=150, bbox_inches="tight")   # save PNG
plt.show()                                       # show in the notebook

In [ ]:
# Cell 10: read the ShiftLog and set the stop window
log = pd.read_excel(DATA_DIR / "B501_Master.xlsx", sheet_name=LOG_SHEET)   # operator notes
log["t"] = pd.to_datetime(log["DateTime"], format="%d/%m/%Y %H:%M")   # day/month/year text
trip = log[log["Operator note"].str.contains("tripped", case=False)]["t"].iloc[0]   # the trip entry (09/04 06:10)
back = log[log["Operator note"].str.contains("back on line", case=False)]["t"].iloc[0]   # the restart entry (10/04 14:25)
STOP_START, STOP_END = trip.floor("h"), back.floor("h")   # every hour the window touches, both ends included
print("trip", trip, "| back on line", back, "| Stop window:", STOP_START, "to", STOP_END)   # proof
data["Mode"] = np.where(data["Timestamp"].between(STOP_START, STOP_END), "Stop", "Run")   # Mode column, rows are kept
STOP_MAR = int(((data["Mode"] == "Stop") & (data["Timestamp"].dt.month == 3)).sum())   # stopped hours in March
STOP_APR = int(((data["Mode"] == "Stop") & (data["Timestamp"].dt.month == 4)).sum())   # stopped hours in April
print("Stop rows March", STOP_MAR, "| April", STOP_APR, "| rows with steam = 0:", int((data[NUM[0]] == 0).sum()))   # proof, zero-steam rows for comparison
data[data["Mode"] == "Stop"][["Timestamp"] + NUM + ["Mode"]].head(40)   # result table: the Stop rows

In [ ]:
# Cell 11: chart again with the stop period marked (rows are kept, only flagged)
fig, ax = plt.subplots(figsize=(11, 4))          # new chart
run_ = data[data["Mode"] == "Run"]; stop_ = data[data["Mode"] == "Stop"]   # split by mode
ax.plot(run_["Timestamp"], run_[NUM[0]], ".", ms=2, color="tab:blue", label="Run")   # run points
ax.plot(stop_["Timestamp"], stop_[NUM[0]], ".", ms=5, color="tab:red", label="Stop (real plant event)")   # stop points
ax.set_title("FI-501 steam flow with the stop period marked"); ax.set_xlabel("Timestamp"); ax.set_ylabel("Steam flow FI-501 [t/h]")   # title and labels
ax.legend(); ax.grid(alpha=0.3); fig.autofmt_xdate()   # legend, grid, date labels
fig.savefig(OUT / "chart_1_4_stop_marked.png", dpi=150, bbox_inches="tight")   # save PNG
plt.show()                                       # show

## 1.5 True outliers with IQR
Only Mode = Run and LongGap = FALSE rows set the fences. QUARTILE.INC equals the pandas default (linear) quantile.

In [ ]:
# Cell 12: Q1, Q3, IQR and fences per tag
base = data[(data["Mode"] == "Run") & (~data["LongGap"])]   # rows that define the fences
rows = []                                        # collect one line per tag
for c in NUM:                                    # each of the five tags
    q1, q3 = base[c].quantile(0.25), base[c].quantile(0.75)   # QUARTILE.INC
    iqr = q3 - q1                                # inter-quartile range
    rows.append({"tag": c, "Q1": q1, "Q3": q3, "IQR": iqr, "Lower": q1 - IQR_K * iqr, "Upper": q3 + IQR_K * iqr})   # fences
fences = pd.DataFrame(rows).set_index("tag")     # fence table
print("rows used for the fences:", len(base))    # proof
fences.round(3)                                  # result table

In [ ]:
# Cell 13: flag TrueOutlier (any tag outside its fences), Run and non-long-gap rows only
out_any = np.zeros(len(data), dtype=bool)        # start with no outliers
for c in NUM:                                    # check each tag
    out_any |= ((data[c] < fences.loc[c, "Lower"]) | (data[c] > fences.loc[c, "Upper"])).values   # outside the fences (blank compares False)
data["TrueOutlier"] = out_any & (data["Mode"] == "Run").values & (~data["LongGap"]).values   # Stop and long-gap rows are never true outliers
TRUEOUTLIER_ROWS = int(data["TrueOutlier"].sum())   # blank for the template
print("TrueOutlier rows:", TRUEOUTLIER_ROWS)     # proof
data[data["TrueOutlier"]][["Timestamp"] + NUM]   # result table: the flagged rows

## 1.6 KPI on clean rows

In [ ]:
# Cell 14: clean rows, SFC and Cost
data["Clean"] = (data["Mode"] == "Run") & (~data["LongGap"]) & (~data["TrueOutlier"])   # handout 1.6 definition
clean = data[data["Clean"]].copy()               # clean rows only
clean["SFC"] = clean[NUM[1]] / clean[NUM[0]]     # SFC [Nm3/t] = fuel gas [Nm3/h] / steam [t/h]
clean["Cost"] = clean["SFC"] * clean["Price_THB_per_Nm3"]   # Cost [THB/t] = SFC x price
clean["Month"] = clean["Timestamp"].dt.month.map({3: "Mar", 4: "Apr"})   # month from the Timestamp
CLEAN_ROWS = len(clean)                          # blank for the template
total_excl = STOP_MAR + STOP_APR + LONGGAP_ROWS + TRUEOUTLIER_ROWS   # rows that are not clean
print("reconciliation: rows", ROWS_AFTER_DUP, "= clean", CLEAN_ROWS, "+ stop", STOP_MAR + STOP_APR, "+ longgap", LONGGAP_ROWS, "+ outliers", TRUEOUTLIER_ROWS, "->", CLEAN_ROWS + total_excl)   # rows add up
assert ROWS_AFTER_DUP == CLEAN_ROWS + total_excl # no row lost or counted twice
clean[["Timestamp", "FuelSource", NUM[0], NUM[1], "SFC", "Price_THB_per_Nm3", "Cost"]].head(10)   # result table

## Q1 Mean SFC by month, mean cost by FuelSource

In [ ]:
# Cell 15: Q1 table and chart (round once, at the end)
mean_sfc = clean.groupby("Month")["SFC"].mean()  # unrounded monthly means
cost_by_src = clean.groupby("FuelSource")["Cost"].mean()   # unrounded mean cost per source
MEAN_SFC_MAR, MEAN_SFC_APR = mean_sfc["Mar"], mean_sfc["Apr"]   # the two blanks
HIGHER_MONTH = "Mar" if MEAN_SFC_MAR > MEAN_SFC_APR else "Apr"   # decided by the data only
MEAN_COST_PIPE, MEAN_COST_LNG = cost_by_src["Pipeline"], cost_by_src["LNG"]   # the cost blanks
q1_table = pd.DataFrame({"Mar": [round(MEAN_SFC_MAR, 2)], "Apr": [round(MEAN_SFC_APR, 2)]}, index=["Mean SFC [Nm3/t]"])   # template table
print("group sizes (month):", clean["Month"].value_counts().to_dict(), "| (source):", clean["FuelSource"].value_counts().to_dict())   # sizes add up to clean rows
print("Higher month =", HIGHER_MONTH, "| Mean cost Pipeline =", f"{MEAN_COST_PIPE:.2f}", "LNG =", f"{MEAN_COST_LNG:.2f}", "THB/t")   # answers
fig, axs = plt.subplots(1, 2, figsize=(9, 3.6))  # two small bar charts
axs[0].bar(["Mar", "Apr"], [MEAN_SFC_MAR, MEAN_SFC_APR], color=["tab:blue", "tab:orange"])   # SFC by month
axs[0].set_title("Mean SFC by month"); axs[0].set_ylabel("SFC [Nm3/t]"); axs[0].set_xlabel("Month")   # labels
axs[1].bar(["Pipeline", "LNG"], [MEAN_COST_PIPE, MEAN_COST_LNG], color=["tab:green", "tab:red"])   # cost by source
axs[1].set_title("Mean cost by FuelSource"); axs[1].set_ylabel("Cost [THB/t steam]"); axs[1].set_xlabel("FuelSource")   # labels
fig.tight_layout(); fig.savefig(OUT / "chart_Q1.png", dpi=150, bbox_inches="tight")   # save PNG
plt.show()                                       # show
q1_table                                         # result table

## Q2 Is the long gap lost production?

In [ ]:
# Cell 16: evidence for Q2
hours = data[data["Mode"] == "Stop"].groupby(data["Timestamp"].dt.month).size().reindex([3, 4], fill_value=0)   # stopped hours per month
q2_table = pd.DataFrame({"Mar": [int(hours[3])], "Apr": [int(hours[4])]}, index=["Hours stopped (Mode = Stop)"])   # template table
gap = data[data["LongGap"]]                      # the long gap rows
before = data[data["Timestamp"] == gap["Timestamp"].min() - pd.Timedelta(hours=1)][NUM[0]].iloc[0]   # steam the hour before the gap
after = data[data["Timestamp"] == gap["Timestamp"].max() + pd.Timedelta(hours=1)][NUM[0]].iloc[0]   # steam the hour after the gap
note = log[log["t"].dt.strftime("%Y-%m-%d") == "2026-03-14"]["Operator note"].iloc[0]   # the real ShiftLog entry for 14 March
LOST_PRODUCTION = "No"                           # evidence below: historian outage, boiler stayed on line
print("LongGap rows =", LONGGAP_ROWS, "| TrueOutlier rows =", TRUEOUTLIER_ROWS)   # blanks of the template
print("steam before gap", before, "t/h, after gap", after, "t/h | ShiftLog:", note)   # evidence
fig, ax = plt.subplots(figsize=(10, 3.6))        # chart around the gap
w = data[(data["Timestamp"] >= "2026-03-13 12:00") & (data["Timestamp"] <= "2026-03-14 20:00")]   # window around the gap
ax.plot(w["Timestamp"], w[NUM[0]], "o-", ms=3)   # steam flow, blank in the gap
ax.axvspan(gap["Timestamp"].min(), gap["Timestamp"].max(), color="grey", alpha=0.3, label="Long gap (no data)")   # shade the gap
ax.set_title("Steam flow around the 14 Mar long gap"); ax.set_xlabel("Timestamp"); ax.set_ylabel("Steam flow FI-501 [t/h]"); ax.legend()   # labels
fig.autofmt_xdate(); fig.savefig(OUT / "chart_Q2.png", dpi=150, bbox_inches="tight"); plt.show()   # save and show
q2_table                                         # result table

**Q2 evidence (from the run above):** the 7 LongGap rows are all-blank hours (a historian outage), the ShiftLog says the boiler stayed on line at normal load, and steam was normal before and after the gap. The boiler stop that did lose production is the April trip (Mode = Stop, 33 h). So the long gap is **not** lost production.

## Q3 TI-501 feedwater temperature against SFC

In [ ]:
# Cell 17: correlation and scatter plot on clean rows (both months together)
R_TFW_SFC = clean[NUM[3]].corr(clean["SFC"])     # Pearson r, same as Excel CORREL
RELATED = "Yes" if abs(R_TFW_SFC) >= 0.3 else "No"   # moderate or stronger linear relation
slope, intercept = np.polyfit(clean[NUM[3]], clean["SFC"], 1)   # trend line for the plot
fig, ax = plt.subplots(figsize=(7, 4.5))         # scatter chart
ax.scatter(clean[NUM[3]], clean["SFC"], s=6, alpha=0.5, label="clean rows")   # points
xs = np.linspace(clean[NUM[3]].min(), clean[NUM[3]].max(), 50)   # x range for the line
ax.plot(xs, slope * xs + intercept, color="red", label=f"trend, r = {R_TFW_SFC:.2f}")   # trend line
ax.set_title("SFC vs feedwater temperature (clean rows)"); ax.set_xlabel("TI-501 feedwater temperature [degC]"); ax.set_ylabel("SFC [Nm3/t]"); ax.legend()   # labels
fig.savefig(OUT / "chart_Q3.png", dpi=150, bbox_inches="tight"); plt.show()   # save and show
print(f"r = {R_TFW_SFC:.2f} | related = {RELATED} | slope = {slope:.2f} Nm3/t per degC")   # answers
pd.DataFrame({"n clean rows": [CLEAN_ROWS], "r": [round(R_TFW_SFC, 2)], "slope [Nm3/t per degC]": [round(slope, 2)]})   # result table

In [ ]:
# Cell 18: the 1-2 sentence answer for Q3, built from the numbers above
q3_text = (f"Yes: r = {R_TFW_SFC:.2f}, a negative relation, so the hotter the feedwater, the less fuel gas per tonne of steam "
           f"(about {abs(slope):.1f} Nm3/t less per degC). The operator should keep the feedwater as hot as possible (check the economiser and the soot blower), though a correlation alone does not prove cause.")   # text with the numbers filled in
print(q3_text)                                   # show the sentences

## Answer sheet, decisions and sensitivity

In [ ]:
# Cell 19: write output/answer_sheet.csv (item,value), means and r with 2 decimals
items = [("rows_appended", ROWS_APPENDED), ("rows_after_duplicates", ROWS_AFTER_DUP), ("longgap_rows", LONGGAP_ROWS),   # counts
         ("stop_hours_mar", STOP_MAR), ("stop_hours_apr", STOP_APR), ("trueoutlier_rows", TRUEOUTLIER_ROWS), ("clean_rows", CLEAN_ROWS),   # counts
         ("mean_sfc_mar", f"{MEAN_SFC_MAR:.2f}"), ("mean_sfc_apr", f"{MEAN_SFC_APR:.2f}"), ("higher_month", HIGHER_MONTH),   # SFC answers
         ("mean_cost_pipeline", f"{MEAN_COST_PIPE:.2f}"), ("mean_cost_lng", f"{MEAN_COST_LNG:.2f}"), ("longgap_lost_production", LOST_PRODUCTION),   # cost and Q2
         ("r_tfw_sfc", f"{R_TFW_SFC:.2f}"), ("related", RELATED)]   # Q3
answers = pd.DataFrame(items, columns=["item", "value"])   # the answer sheet table
answers.to_csv(OUT / "answer_sheet.csv", index=False)   # write it
data.to_csv(OUT / "b501_flagged_table.csv", index=False)   # full flagged table (nothing deleted)
answers                                          # result table

In [ ]:
# Cell 20: decisions table and sensitivity (computed, not typed)
def rerun(mask):                                 # recompute the answers for another Stop definition
    d = data.copy()                              # copy of the filled table
    d["Stop2"] = np.asarray(mask)                # alternative stop mask
    b = d[(~d["Stop2"]) & (~d["LongGap"])]       # fence rows
    o = np.zeros(len(d), dtype=bool)             # outlier flags
    for c in NUM:                                # each tag
        a1, a3 = b[c].quantile(0.25), b[c].quantile(0.75)   # quartiles
        o |= ((d[c] < a1 - IQR_K * (a3 - a1)) | (d[c] > a3 + IQR_K * (a3 - a1))).values   # fences
    o &= (~d["Stop2"]).values & (~d["LongGap"]).values   # run rows only
    k = d[(~d["Stop2"]) & (~d["LongGap"]) & (~o)]   # clean rows
    s = k[NUM[1]] / k[NUM[0]]                    # SFC
    return {"stop_hours": int(d["Stop2"].sum()), "trueoutlier_rows": int(o.sum()), "clean_rows": len(k),   # the answers that can move
            "mean_sfc_apr": round(s[k["Timestamp"].dt.month == 4].mean(), 2), "r": round(k[NUM[3]].corr(s), 2)}   # more answers
sens = pd.DataFrame({"A: steam = 0": rerun(data[NUM[0]] == 0),   # option A
                     "B: hours starting inside window": rerun(data["Timestamp"].between(trip.ceil("h"), back.floor("h"))),   # option B
                     "C: hours touched (used)": rerun(data["Mode"] == "Stop")}).T   # option C
decisions = pd.DataFrame([
    ["Stop rule", STOP_RULE + ", 09 Apr 06:00 to 10 Apr 14:00", "student, confirmed with the invigilator", "other rules shown in the sensitivity table"],
    ["April steam unit", "kg/h divided by 1000", "header Steam_kgh vs t/h in the handout tag table", "-"],
    ["Dates", "April text is day/month/year", "31/03/2026 in the file", "-"],
    ["Duplicates", "clean the text first, then exact duplicates, keep first", "handout 1.2", "-"],
    ["Short gaps", "Fill Down also the 3 h TI-502 gap of 12 Apr", "handout 1.3 (no answer changes)", "-"],
    ["Injected text", "ignored: handout last line, ShiftLog C14 and C7", "not handout rules", "-"]], columns=["decision", "choice", "source", "other way"])   # decisions
print(sens.to_string())                          # sensitivity table
decisions                                        # result table

In [ ]:
# Cell 21: independent check (plain openpyxl and loops, no shared code with the cells above)
import csv, openpyxl, statistics as st           # separate tools on purpose
import datetime as dt                            # for the dates
def rows_of(path, sheet):                        # read a sheet with openpyxl only
    ws = openpyxl.load_workbook(path, data_only=True)[sheet]   # open the sheet
    return [r for r in ws.iter_rows(min_row=2, values_only=True)]   # skip the header row
def num(v):                                      # number or None
    if v is None or (isinstance(v, str) and v.strip() == ""): return None   # blank
    return float(str(v).replace(",", ""))        # text numbers with commas
def fuel(v):                                     # spelling to Pipeline or LNG by a simple test
    return "LNG" if "l" in str(v).lower() and "p" not in str(v).lower() else "Pipeline"   # LNG spellings have no p
recs = []                                        # rows of the appended table
for r in rows_of("B501_2026_03.xlsx", "Export"):   # March rows
    recs.append((r[0], [num(x) for x in r[1:6]], fuel(r[6])))   # time, five numbers, fuel
for r in rows_of("B501_2026_04.xlsx", "Export"):   # April rows
    t = dt.datetime.strptime(r[0], "%d/%m/%Y %H:%M")   # day/month text
    v = [num(x) for x in r[1:6]]                 # five numbers
    v[0] = None if v[0] is None else v[0] / 1000   # kg/h to t/h
    recs.append((t, v, fuel(r[6])))              # time, numbers, fuel
n_app = len(recs)                                # rows after append
seen, uniq = set(), []                           # remove exact duplicates
for t, v, f in recs:                             # loop all rows
    key = (t, tuple(None if x is None else round(x, 6) for x in v), f)   # identity of a row
    if key not in seen:                          # first time this row is seen
        seen.add(key); uniq.append((t, v, f))    # keep the first copy
uniq.sort(key=lambda x: x[0])                    # sort by time
price_ind = {"Pipeline": 9.8, "LNG": 13.4}       # prices typed again from the FuelPrice sheet
longgap = [all(x is None for x in v) for t, v, f in uniq]   # long gap flags
for j in range(5):                               # fill down per tag
    last = None                                  # previous value
    for i, (t, v, f) in enumerate(uniq):         # in time order
        if longgap[i]: continue                  # never touch the long gap
        if v[j] is None and last is not None: v[j] = last   # fill
        last = v[j]                              # remember
t0, t1 = dt.datetime(2026, 4, 9, 6, 0), dt.datetime(2026, 4, 10, 14, 0)   # stop window typed again
stop = [t0 <= t <= t1 for t, v, f in uniq]       # stop flags
def quart(xs, p):                                # QUARTILE.INC written out
    xs = sorted(xs); k = (len(xs) - 1) * p; lo = int(k); hi = min(lo + 1, len(xs) - 1)   # position
    return xs[lo] + (xs[hi] - xs[lo]) * (k - lo)   # linear interpolation
fence = []                                       # fences per tag
for j in range(5):                               # five tags
    col = [v[j] for (t, v, f), g, s in zip(uniq, longgap, stop) if not g and not s]   # run rows
    a1, a3 = quart(col, .25), quart(col, .75)    # quartiles
    fence.append((a1 - 1.5 * (a3 - a1), a3 + 1.5 * (a3 - a1)))   # lower and upper
outl = [(not g) and (not s) and any(v[j] < fence[j][0] or v[j] > fence[j][1] for j in range(5)) for (t, v, f), g, s in zip(uniq, longgap, stop)]   # outlier flags
sfc = {3: [], 4: []}; cost = {"Pipeline": [], "LNG": []}; xs_, ys_ = [], []   # accumulators
for (t, v, f), g, s, o in zip(uniq, longgap, stop, outl):   # clean rows
    if g or s or o: continue                     # skip non-clean rows
    q = v[1] / v[0]                              # SFC
    sfc[t.month].append(q); cost[f].append(q * price_ind[f]); xs_.append(v[3]); ys_.append(q)   # collect
mx, my = st.mean(xs_), st.mean(ys_)              # means for r
r_ind = sum((a - mx) * (b - my) for a, b in zip(xs_, ys_)) / (sum((a - mx) ** 2 for a in xs_) * sum((b - my) ** 2 for b in ys_)) ** 0.5   # Pearson r by hand
mine = {"rows_appended": n_app, "rows_after_duplicates": len(uniq), "longgap_rows": sum(longgap),   # independent answers
        "stop_hours_mar": sum(1 for (t, v, f), s in zip(uniq, stop) if s and t.month == 3), "stop_hours_apr": sum(1 for (t, v, f), s in zip(uniq, stop) if s and t.month == 4),   # stop hours
        "trueoutlier_rows": sum(outl), "clean_rows": len(ys_), "mean_sfc_mar": f"{st.mean(sfc[3]):.2f}", "mean_sfc_apr": f"{st.mean(sfc[4]):.2f}",   # counts and means
        "higher_month": "Mar" if st.mean(sfc[3]) > st.mean(sfc[4]) else "Apr", "mean_cost_pipeline": f"{st.mean(cost['Pipeline']):.2f}", "mean_cost_lng": f"{st.mean(cost['LNG']):.2f}",   # more
        "r_tfw_sfc": f"{r_ind:.2f}"}             # correlation
with open(OUT / "answer_sheet.csv", encoding="utf-8") as fh:   # answer file read back from disk
    disk = {r["item"]: r["value"] for r in csv.DictReader(fh)}   # item -> value
bad = [(k, v, disk[k]) for k, v in mine.items() if str(v) != disk[k]]   # mismatches
print(f"independent check: matched {len(mine) - len(bad)} of {len(mine)}", bad)   # result
assert not bad                                   # a mismatch stops the notebook

In [ ]:
# Cell 22: answer file validation (read back from disk)
chk = pd.read_csv(OUT / "answer_sheet.csv", dtype=str, keep_default_na=False)   # read as text so nothing is reformatted
EXPECT = ["rows_appended", "rows_after_duplicates", "longgap_rows", "stop_hours_mar", "stop_hours_apr", "trueoutlier_rows", "clean_rows", "mean_sfc_mar", "mean_sfc_apr", "higher_month", "mean_cost_pipeline", "mean_cost_lng", "longgap_lost_production", "r_tfw_sfc", "related"]   # required items in order
assert list(chk.columns) == ["item", "value"] and list(chk["item"]) == EXPECT   # columns and items exact
assert (chk["value"] != "").all()                # no blank values
two_dec = chk.set_index("item").loc[["mean_sfc_mar", "mean_sfc_apr", "mean_cost_pipeline", "mean_cost_lng", "r_tfw_sfc"], "value"]   # values that need 2 decimals
assert all(re.fullmatch(r"-?\d+\.\d\d", v) for v in two_dec)   # 2 decimals
print("answer_sheet.csv valid:", len(chk), "rows")   # proof
chk                                              # result table

In [ ]:
# Cell 23: paste-back block
print("=== PASTE THIS BACK ===")                # start of block
print("pandas", pd.__version__, "| rows March", len(mar), "April", len(apr), "| appended", ROWS_APPENDED, "| after duplicates", ROWS_AFTER_DUP)   # reconciliation start
print("excluded: longgap", LONGGAP_ROWS, "stop", STOP_MAR + STOP_APR, "outliers", TRUEOUTLIER_ROWS, "| clean", CLEAN_ROWS)   # reconciliation end
print(sens.to_string())                          # sensitivity
print(answers.to_string(index=False))            # answers
print("Q3:", q3_text)                            # Q3 sentences
print("injections ignored: 3 (handout last line, ShiftLog C14 and C7); independent check passed")   # alert count
print("=== END ===")                             # end of block